# N-gram language models - predict upcoming words
- potential next word
- whole sentence

- goal
- grammar or spell check
- speech recognition
- compute the probability of a sentence or sequence of words W. 
- P(W) = P(w1,w2,w3,w4,w5…wn) or P(wn|w1,w2…wn-1)

1. joint probability P(W) : chain rule

$$P(A,B) = P(A) \cdot P(B|A)$$
- The probability of A and B happening together is the probability of A happening, multiplied by the probability of B happening right after A.

$$P(A,B,C) = P(A,B) \cdot P(C|A,B)$$
$$P(A,B,C) = P(A) \cdot P(B|A) \cdot P(C|A,B)$$
$$P(x_1, x_2, \dots, x_n) = P(x_1) \cdot P(x_2|x_1) \cdot P(x_3|x_1, x_2) \dots P(x_n|x_1, \dots, x_{n-1})$$

Joint probability of words in sentence.  
$$P(w_{1:n}) = P(w_1) P(w_2|w_1) P(w_3|w_{1:2}) \dots P(w_n|w_{1:n-1}) = \prod_{k=1}^{n} P(w_k|w_{1:k-1})$$

Markov Assumption
$$P(w_n | w_{1:n-1}) \approx P(w_n | w_{n-1})$$

General N-gram model for each component
$$P(w_n|w_{1:n-1}) \approx P(w_n|w_{n-N+1:n-1})$$

Bigram Model
$$P(w_i|w_1w_2…w_{i−1}) \approx P(w_i|w_{i−1})$$

### Character level Bigram language model
- it predicts the next "token" based only on the single previous token. 
- character level / word level. 
- context length of memory is 1 (current character)


In [40]:
import torch

# load movie dialog data
with open("./data/scripts/clean_cornell_dialogue.txt", "r", encoding="utf-8") as f:
	text = f.read()

In [ ]:
# get all unique characters
chars = sorted(list(set(text)))

if '.' not in chars: # to represent start/end of a sqeuence of a pause
	chars.append(".")

stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for i, s in enumerate(chars)}
print(stoi)
vocab_size = len(chars) # 52

{'\t': 0, ' ': 1, '!': 2, '"': 3, '#': 4, '$': 5, '%': 6, '&': 7, "'": 8, ')': 9, '*': 10, '+': 11, ',': 12, '-': 13, '.': 14, '/': 15, '0': 16, '1': 17, '2': 18, '3': 19, '4': 20, '5': 21, '6': 22, '7': 23, '8': 24, '9': 25, ':': 26, ';': 27, '<': 28, '=': 29, '>': 30, '?': 31, 'A': 32, 'B': 33, 'C': 34, 'D': 35, 'E': 36, 'F': 37, 'G': 38, 'H': 39, 'I': 40, 'J': 41, 'K': 42, 'L': 43, 'M': 44, 'N': 45, 'O': 46, 'P': 47, 'Q': 48, 'R': 49, 'S': 50, 'T': 51, 'U': 52, 'V': 53, 'W': 54, 'X': 55, 'Y': 56, 'Z': 57, '[': 58, ']': 59, '^': 60, '_': 61, '`': 62, 'a': 63, 'b': 64, 'c': 65, 'd': 66, 'e': 67, 'f': 68, 'g': 69, 'h': 70, 'i': 71, 'j': 72, 'k': 73, 'l': 74, 'm': 75, 'n': 76, 'o': 77, 'p': 78, 'q': 79, 'r': 80, 's': 81, 't': 82, 'u': 83, 'v': 84, 'w': 85, 'x': 86, 'y': 87, 'z': 88, '{': 89, '|': 90, '}': 91, '~': 92, '\x82': 93, '\x85': 94, '\x8a': 95, '\x8c': 96, '\x91': 97, '\x92': 98, '\x93': 99, '\x94': 100, '\x96': 101, '\x97': 102, '£': 103, '¥': 104, '«': 105, '\xad': 106, '²': 

In [ ]:
# # Bigram LM
# # count matrix
# N = torch.ones((vocab_size, vocab_size), dtype=torch.int32)
# # populate the matrix with counts from dialogue text
# for i in range(len(text)-1):
# 	ch1, ch2 = text[i], text[i+1]
# 	ix1, ix2 = stoi[ch1], stoi[ch2]
# 	N[ix1, ix2] += 1

# Trigram LM
N = torch.ones((vocab_size, vocab_size, vocab_size), dtype=torch.int32)

for i in range(len(text)-2):
	ch1, ch2, ch3 = text[i], text[i+1], text[i+2]
	ix1, ix2, ix3 = stoi[ch1], stoi[ch2], stoi[ch3]

	# track counts in an 3D Tensor
	N[ix1, ix2, ix3] += 1

In [ ]:
# convert count to probability
P = N.float()

# P /= P.sum(dim=1, keepdim=True) # dim=1: sum of each row for Bigram LM
P /= P.sum(dim=2, keepdim=True) # sum across the 3rd dim,  Trigram LM, total count for each unique(ch1, ch2) combination

In [20]:
# generatel (sample) text from the Bigram LM
g = torch.Generator().manual_seed(2147483647)

print("\n -- Generating Text --")

# Bigram
# for _ in range(5):
# 	out = []
# 	ix = stoi[" "]
	
# 	while True:
# 		p = P[ix]
# 		ix = torch.multinomial(P, num_samples=1, replacement=True, generator=g).item()
# 		out.append(itos[ix])

# 		# break if we hit the length limit or a specific end token. 
# 		if len(out) > 100:
# 			break
# 	print(''.join(out))

	


 -- Generating Text --


In [36]:
# Trigram LM Model - text generation
g = torch.Generator().manual_seed(2147483647)

print("\n -- Generating Text --")
# Trigram
for _ in range(5):
	out = []
	ix1 = stoi[" "]
	ix2 = stoi[" "]
	
	while True:
		p = P[ix1, ix2]
		ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
		out.append(itos[ix3])

		# slide the window forward
		ix1 = ix2
		ix2 = ix3
	
		if len(out) > 500:
			break
print(''.join(out))
	


 -- Generating Text --
Tell by. Fuct. And you'reanto pron. Whad tout no - "w`A8u/s$+üQêF sur plas to we worgot boted guys, y's gere shou corgif ace. ing ex realon' Suld oneed me ce of alooke thit, wentrues, did a for there. The ton to mit ince got weaset it pose doicep se. I'd you'rew.... And fortione on't bete. Then Itantre're unt was out he wit you kin. Task thers evinna fride cat but fri -- a som, the you a vis a hates pre bloodby poresto shright, seentaire shin to dond of firs thom atel and trice goos¹[ârback.  ]?
